In [2]:
from src import *

import numpy as np
import pandas as pd

from typing import List

# READ
> Don't try to actually run all cells. It would take too long from the delay added <br>
> Just choose which segments require testing.
>
> Reduce `MINIMUM_DELAY` at your own risk of being blocked

> To perform scraping of `Details`, `StatisticsSummary`, `StatisticsSubmissions`, ensure there's a `csv` already in `Season/`.<br>
> Otherwise run `AutomateSeason()` in the `Seasons` segment to generate the database of anime links.

In [2]:
Scraper.MINIMUM_DELAY = 10

# Seasons

In [3]:
assert False

AssertionError: 

In [ ]:
def AutomateSeason(YEARS : List[str]):
    for YEAR in YEARS:
        for SEASON in Scraper.SEASONS.__members__.values():
            url = Scraper.build_URL_Seasons(YEAR, SEASON)
            print(url)
    
            try:
                seasonalSoup = Scraper.scrape(url)
                df_Season = pd.DataFrame(SeasonPage.parse(seasonalSoup))
    
                filename = f'Season/{YEAR}_{SEASON.value}.csv'
                df_Season.to_csv(filename)
                print(filename)
    
            except Exception as e:
                print(url, e)

## 1. Scrape All 1996 Seasons Record

In [ ]:
AutomateSeason(np.arange(1995, 1996))

## 2. Scrape Modern Anime (2020-2025)

In [ ]:
AutomateSeason(np.arange(2020, 2026))

In [ ]:
AutomateSeason(np.arange(1996, 2020))

# Details

In [ ]:
assert False

## 1. Scrape [First 2, 2 Random, and Last 2] Anime from 1995 Fall

In [ ]:
SEASON = Scraper.SEASONS.FALL.value
YEAR = 1995

In [ ]:
df = pd.read_csv(f'Season/{YEAR}_{SEASON}.csv', index_col=0)
df_Target = df.iloc[[0,1,
                np.random.randint(2,df.shape[0]-2),
                np.random.randint(2,df.shape[0]-2),
                -2,-1]].copy()

df_Target

In [ ]:
df_Details = None

for _, row in df_Target.iterrows():
    output = {}
    newRow = None
    
    try:
        soup = Scraper.scrape(row.Link)
        output = DetailsPage.parse(soup)
        output['Title'] = row.Title

        newRow = pd.DataFrame([output])
        
        if df_Details is None:
            df_Details = newRow
        else:
            df_Details = pd.concat([df_Details, newRow], ignore_index=True)
        
        print(row.Title)
    except Exception as e:
        print(f'ERROR: {row.Link} {e}')

In [ ]:
filename = f'Detail/{YEAR}_{SEASON}_sample.csv'
df_Details.to_csv(filename)

## 2. Scrape All Entries From Specific Year and Season

In [7]:
def AutomateDetails(YEAR : int | str, SEASON : Scraper.SEASONS):
    df_Details = None

    for _, row in pd.read_csv(f'Season/{YEAR}_{SEASON.value}.csv').iterrows():
        output = {}
        newRow = None
        
        try:
            soup = Scraper.scrape(row.Link)
            output = DetailsPage.parse(soup)
            output['Title'] = row.Title
    
            newRow = pd.DataFrame([output])
            
            if df_Details is None:
                df_Details = newRow
            else:
                df_Details = pd.concat([df_Details, newRow], ignore_index=True)
            
            print(row.Title)
        except Exception as e:
            print(f'ERROR: {row.Link} {e}')    
        
    filename = f'Detail/{YEAR}_{SEASON.value}.csv'
    df_Details.to_csv(filename)
    print(filename)

In [ ]:
AutomateDetails(2024, Scraper.SEASONS.FALL)

In [12]:
#for SEASON in Scraper.SEASONS.__members__.values():
for SEASON in [Scraper.SEASONS.SPRING, Scraper.SEASONS.SUMMER, Scraper.SEASONS.WINTER]:
    AutomateDetails(2023, SEASON)

Kimetsu no Yaiba: Katanakaji no Sato-hen
[Oshi no Ko]
Jigokuraku
Mashle
Tengoku Daimakyou
Dr. Stone: New World
Yamada-kun to Lv999 no Koi wo Suru
Boku no Kokoro no Yabai Yatsu
Kono Subarashii Sekai ni Bakuen wo!
Isekai de Cheat Skill wo Te ni Shita Ore wa, Genjitsu Sekai wo mo Musou Suru: Level Up wa Jinsei wo Kaeta
Skip to Loafer
Tonikaku Kawaii 2nd Season
Dead Mount Death Play
Kimi wa Houkago Insomnia
Kaminaki Sekai no Kamisama Katsudou
Mahoutsukai no Yome Season 2
Tensei Kizoku no Isekai Boukenroku: Jichou wo Shiranai Kamigami no Shito
Isekai Shoukan wa Nidome desu
Kanojo ga Koushaku-tei ni Itta Riyuu
Otonari ni Ginga
Isekai wa Smartphone to Tomo ni. 2
Ousama Ranking: Yuuki no Takarabako
Megami no Café Terrace
Yuusha ga Shinda!
My Home Hero
Isekai One Turn Kill Neesan: Ane Douhan no Isekai Seikatsu Hajimemashita
Kidou Senshi Gundam: Suisei no Majo Season 2
Niehime to Kemono no Ou
Kuma Kuma Kuma Bear Punch!
Mahou Shoujo Magical Destroyers
Edens Zero 2nd Season
Jijou wo Shiranai Tenko

# Statistics [1] Summary and [2] Submissions

In [ ]:
assert False

## 1. Get the Scores of Every Anime in Season and Year

In [3]:
def AutomateStatSummary(YEAR:int, SEASONS:Scraper.SEASONS) -> pd.DataFrame:
    df_StatSummary = None
    
    for _, row in pd.read_csv(f'Season/{YEAR}_{SEASON.value}.csv', index_col=0).iterrows():
        URL = f'{row.Link}/stats'
        print(URL)
    
        try:
            soup = Scraper.scrape(URL)
            data = StatisticsSummaryPage.parse(soup)
            data['Title'] = row.Title
        
            if df_StatSummary is None: 
                df_StatSummary = pd.DataFrame([data])
            else:
                df_StatSummary = pd.concat([df_StatSummary, pd.DataFrame([data])], ignore_index=True)
        except Exception as e:
            print(f'ERROR URL {e}')
        
    filename = f'Statistics/Summary/{YEAR}_{SEASON.value}.csv'
    print(filename)
    df_StatSummary.to_csv(filename)
    
    return df_StatSummary

In [4]:
#YEAR = 2024
#SEASON = Scraper.SEASONS.FALL

for YEAR in np.arange(2022, 2024):
    #for SEASON in [Scraper.SEASONS.WINTER]: 
    for SEASON in Scraper.SEASONS.__members__.values():
        print(YEAR, SEASON)
        AutomateStatSummary(YEAR, SEASON)

2022 SEASONS.FALL
https://myanimelist.net/anime/44511/Chainsaw_Man/stats
https://myanimelist.net/anime/50602/Spy_x_Family_Part_2/stats
https://myanimelist.net/anime/49596/Blue_Lock/stats
https://myanimelist.net/anime/50172/Mob_Psycho_100_III/stats
https://myanimelist.net/anime/48316/Kage_no_Jitsuryokusha_ni_Naritakute/stats
https://myanimelist.net/anime/49918/Boku_no_Hero_Academia_6th_Season/stats
https://myanimelist.net/anime/47917/Bocchi_the_Rock/stats
https://myanimelist.net/anime/41467/Bleach__Sennen_Kessen-hen/stats
https://myanimelist.net/anime/50425/Fuufu_Ijou_Koibito_Miman/stats
https://myanimelist.net/anime/49891/Tensei_shitara_Ken_deshita/stats
https://myanimelist.net/anime/49709/Fumetsu_no_Anata_e_Season_2/stats
https://myanimelist.net/anime/42962/Uzaki-chan_wa_Asobitai_Double/stats
https://myanimelist.net/anime/52046/Yuusha_Party_wo_Tsuihou_sareta_Beast_Tamer_Saikyoushu_no_Nekomimi_Shoujo_to_Deau/stats
https://myanimelist.net/anime/49784/Mairimashita_Iruma-kun_3rd_Season/st

## 2. Scrape the Latest Viewership across <= 100 Pages for a Given Title

In [ ]:
def AutomateStatSubmissions(TITLE : str, LINK : str):
    df_Statistics = None
    
    for i in np.arange(0, 75*100, 75):
        try:
            URL = f'{LINK}/stats?show={i}#members'
            print(URL)
    
            soup = Scraper.scrape(URL)
    
            
            if df_Statistics is None:
                df_Statistics = StatisticsSubmissionPage.parseAsDf(soup)
            else:
                df_Statistics = pd.concat([df_Statistics, StatisticsSubmissionPage.parseAsDf(soup)], ignore_index=True)
            
        except Exception as e:
            print(f'END {e}')
            break
    
    filename = f'Statistics/Submissions/{TITLE}.csv'
    df_Statistics.to_csv(filename)
    print(filename)

### Try EVA

In [ ]:
EVA = pd.read_csv('Season/1995_fall.csv', index_col=0).loc[0]

AutomateStatSubmissions(TITLE=EVA.Title, LINK=EVA.Link)